<a href="https://colab.research.google.com/github/thiendang184-droid/Robot-AI-homework/blob/main/Bai_23_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import osmnx as ox
import networkx as nx
import folium
import numpy as np


center_coords = [10.7601, 106.6691]
customer_locate = [10.7633, 106.6650]

np.random.seed(10)

drivers = (np.random.normal(size=(5, 2)) * 0.005 + center_coords).tolist()

G = ox.graph_from_point(center_coords, dist=2000, network_type='drive')


dest_node = ox.distance.nearest_nodes(G, X=customer_locate[1], Y=customer_locate[0])

best_driver_idx = -1
min_dist = float('inf')
final_route_coords = []

start_node_coords = None


for i, d_pos in enumerate(drivers):

    orig_node = ox.distance.nearest_nodes(G, X=d_pos[1], Y=d_pos[0])
    try:

        route_len = nx.shortest_path_length(G, orig_node, dest_node, weight='length')

        if route_len < min_dist:
            min_dist = route_len
            best_driver_idx = i

            best_route = nx.shortest_path(G, orig_node, dest_node, weight='length')
            final_route_coords = [(G.nodes[n]['y'], G.nodes[n]['x']) for n in best_route]

            start_node_coords = (G.nodes[orig_node]['y'], G.nodes[orig_node]['x'])
    except:
        continue

m = folium.Map(location=center_coords, zoom_start=15)
folium.Marker(customer_locate, popup="Khách hàng",
              icon=folium.Icon(color='blue', icon='user', prefix='fa')).add_to(m)

for i, d_pos in enumerate(drivers):
    if i == best_driver_idx:
        folium.Marker(start_node_coords, popup=f"XE ĐƯỢC CHỌN (Khớp với đường)",
                      icon=folium.Icon(color='green', icon='car', prefix='fa')).add_to(m)
    else:
        folium.Marker(d_pos, popup=f"Xe {i+1}",
                      icon=folium.Icon(color='lightgray', icon='car', prefix='fa')).add_to(m)

if final_route_coords:
    folium.PolyLine(final_route_coords, color='red', weight=5, opacity=0.8).add_to(m)
m